# Laboratorio 01
## capa silver

### Celda 3. Leer los datos y crear Bronze
En Bronze conservaremos los campos originales como cadena-silvers de texto, sin aplicar todavía las reglas de calidad.
¿Por qué guardamos Bronze antes de limpiar? Porque así conservamos evidencia de los datos recibidos y podemos volver a ejecutar las transformaciones sin regenerar ni consultar la fuente.

In [12]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

spark = (
    SparkSession.builder
    .appName("MusicDiscoveryLab-01")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

BASE_PATH = Path("/home/jovyan/data")

PATHS = {
    "raw": BASE_PATH / "raw",
    "bronze": BASE_PATH / "bronze" / "songs",
    "silver": BASE_PATH / "silver" / "songs",
    "gold": BASE_PATH / "gold" / "genre_metrics",
    "quarantine": BASE_PATH / "quarantine" / "songs",
}

raw_file = PATHS["raw"] / "songs.csv"

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Directorio de datos:", BASE_PATH)
print(f"Archivo entrada: {raw_file}")

Spark: 4.2.0
Master: local[*]
Directorio de datos: /home/jovyan/data
Archivo entrada: /home/jovyan/data/raw/songs.csv


In [13]:
raw_schema = StructType([
    StructField("source_row_id", StringType(), False),
    StructField("song_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("bpm", StringType(), True),
    StructField("energy", StringType(), True),
    StructField("valence", StringType(), True),
    StructField("instrumentalness", StringType(), True),
])

raw_df = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv(str(raw_file))
)

raw_df.write.mode("overwrite").parquet(str(PATHS["bronze"]))

bronze_df = spark.read.parquet(str(PATHS["bronze"]))

print("Registros Bronze:", bronze_df.count())
bronze_df.show(5, truncate=False)

Registros Bronze: 12
+-------------+-------+--------------+--------+----------+---+------+-------+----------------+
|source_row_id|song_id|title         |artist  |genre     |bpm|energy|valence|instrumentalness|
+-------------+-------+--------------+--------+----------+---+------+-------+----------------+
|001          |S001   |Moonlit Waltz |Artist A|Classical |92 |0.42  |0.30   |0.98            |
|002          |S002   |Dark Ballroom |Artist B|Classical |108|0.66  |0.24   |0.95            |
|003          |S003   |Silent Forest |Artist C|Ambient   |70 |0.25  |0.18   |1.00            |
|004          |S004   |Midnight Rain |Artist D|Ambient   |80 |0.32  |0.35   |0.90            |
|005          |S005   |Velvet Shadows|Artist E|Soundtrack|115|0.58  |0.28   |0.96            |
+-------------+-------+--------------+--------+----------+---+------+-------+----------------+
only showing top 5 rows


### Celda 4. Tipar y validar los registros
Definiremos reglas explícitas de calidad:
- song_id, título, artista y género son obligatorios.
- BPM debe ser un número entre 1 y 300.
- energy, valence e instrumentalness deben estar entre 0 y 1.
- Un mismo song_id solo puede aparecer una vez en Silver.
Antes de ejecutar, ajusta el último registro de la celda 2 para que song_id sea "S005" en lugar de "S012". Así tendremos un duplicado intencional de S005.

Como el CSV se genera en cada ejecución, basta con volver a ejecutar las celdas 2 y 3 después de ese cambio.

Fíjate en que usamos cast y después validamos los resultados. Un BPM que no pueda convertirse a entero se transforma en NULL y se identifica como inválido.

In [18]:
typed_df = (
    bronze_df
    .withColumn("song_id", F.trim("song_id"))
    .withColumn("title", F.trim("title"))
    .withColumn("artist", F.trim("artist"))
    .withColumn("genre", F.trim("genre"))
    .withColumn("bpm", F.col("bpm").cast(IntegerType()))
    .withColumn("energy", F.col("energy").cast(DoubleType()))
    .withColumn("valence", F.col("valence").cast(DoubleType()))
    .withColumn(
        "instrumentalness",
        F.col("instrumentalness").cast(DoubleType())
    )
)

reasons = F.filter(
    F.array(
        F.when(
            F.col("song_id").isNull() | (F.col("song_id") == ""),
            F.lit("missing_song_id")
        ).otherwise(F.lit("")),
        F.when(
            F.col("title").isNull() | (F.col("title") == ""),
            F.lit("missing_title")
        ).otherwise(F.lit("")),
        F.when(
            F.col("artist").isNull() | (F.col("artist") == ""),
            F.lit("missing_artist")
        ).otherwise(F.lit("")),
        F.when(
            F.col("genre").isNull() | (F.col("genre") == ""),
            F.lit("missing_genre")
        ).otherwise(F.lit("")),
        F.when(
            F.col("bpm").isNull()
            | (F.col("bpm") < 1)
            | (F.col("bpm") > 300),
            F.lit("invalid_bpm")
        ).otherwise(F.lit("")),
        *[
            F.when(
                F.col(c).isNull()
                | (F.col(c) < 0.0)
                | (F.col(c) > 1.0),
                F.lit(f"invalid_{c}")
            ).otherwise(F.lit(""))
            for c in ["energy", "valence", "instrumentalness"]
        ],
    ),
    lambda reason: reason != ""
)

validated_df = (
    typed_df
    .withColumn("_reasons", reasons)
    .withColumn(
        "rejection_reason",
        F.concat_ws("; ", F.col("_reasons"))
    )
)

invalid_df = (
    validated_df
    .filter(F.size("_reasons") > 0)
    .withColumn("type_quarantine", F.lit("invalid_value"))
    .select(*validated_df.columns,"type_quarantine")
)

valid_candidates_df = validated_df.filter(F.size("_reasons") == 0)

print("Registros con errores:", invalid_df.count())
invalid_df.select(
    "source_row_id", "song_id", "title", "rejection_reason"
).show(truncate=False)

Registros con errores: 2
+-------------+-------+------------+----------------+
|source_row_id|song_id|title       |rejection_reason|
+-------------+-------+------------+----------------+
|010          |S010   |NULL        |missing_title   |
|011          |S011   |Broken Tempo|invalid_bpm     |
+-------------+-------+------------+----------------+



### Celda 5. Deduplicar y construir Silver
Conservaremos el primer registro válido de cada canción según source_row_id. Los registros duplicados se enviarán a cuarentena en lugar de desaparecer silenciosamente.

In [20]:
window_song = Window.partitionBy("song_id").orderBy("source_row_id")

ranked_df = valid_candidates_df.withColumn(
    "_row_number",
    F.row_number().over(window_song)
)

silver_columns = [
    "song_id",
    "title",
    "artist",
    "genre",
    "bpm",
    "energy",
    "valence",
    "instrumentalness",
]

silver_df = ranked_df.filter(
    F.col("_row_number") == 1
).select(*silver_columns)

duplicate_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .withColumn("rejection_reason", F.lit("duplicate_song_id"))
    .withColumn("type_quarantine", F.lit("duplicate"))
    .select(*ranked_df.columns, "type_quarantine")
)

quarantine_df = (
    invalid_df.select(
        *[c for c in invalid_df.columns if c != "_reasons"]
    )
    .unionByName(
        duplicate_df.select(
            *[c for c in invalid_df.columns if c != "_reasons"]
        )
    )
)

silver_df.write.mode("overwrite").parquet(str(PATHS["silver"]))
quarantine_df.write.mode("overwrite").parquet(str(PATHS["quarantine"]))

silver_df = spark.read.parquet(str(PATHS["silver"]))
quarantine_df = spark.read.parquet(str(PATHS["quarantine"]))

print("Silver:", silver_df.count())
print("Cuarentena:", quarantine_df.count())

silver_df.orderBy("song_id").show(truncate=False)

quarantine_df.select(
    "source_row_id", "song_id", "title", "rejection_reason"
).show(truncate=False)

Silver: 10
Cuarentena: 2
+-------+----------------+--------+----------+---+------+-------+----------------+
|song_id|title           |artist  |genre     |bpm|energy|valence|instrumentalness|
+-------+----------------+--------+----------+---+------+-------+----------------+
|S001   |Moonlit Waltz   |Artist A|Classical |92 |0.42  |0.3    |0.98            |
|S002   |Dark Ballroom   |Artist B|Classical |108|0.66  |0.24   |0.95            |
|S003   |Silent Forest   |Artist C|Ambient   |70 |0.25  |0.18   |1.0             |
|S004   |Midnight Rain   |Artist D|Ambient   |80 |0.32  |0.35   |0.9             |
|S005   |Velvet Shadows  |Artist E|Soundtrack|115|0.58  |0.28   |0.96            |
|S006   |Silver Strings  |Artist F|Soundtrack|100|0.48  |0.45   |0.88            |
|S007   |Crimson Waltz   |Artist G|Classical |130|0.72  |0.4    |0.92            |
|S008   |Echoes of Winter|Artist H|Ambient   |65 |0.2   |0.15   |0.99            |
|S009   |Clockwork Dreams|Artist I|Soundtrack|124|0.75  |0.55 